In [1]:
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

import pandas as pd
import numpy as np
import pickle
import json

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"

train_df = pd.read_parquet(PROCESSED_DIR / "train.parquet")
test_df = pd.read_parquet(PROCESSED_DIR / "test.parquet")

with open(PROCESSED_DIR / "encoders.pkl", "rb") as f:
    encoders = pickle.load(f)
n_users = encoders["n_users"]
n_movies = encoders["n_movies"]

global_mean = train_df["rating"].mean()

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

print(f"Train: {train_df.shape}, Test: {test_df.shape}")
print(f"Users: {n_users}, Movies: {n_movies}")

Train: (802553, 6), Test: (197656, 6)
Users: 6040, Movies: 3706


In [2]:
user_factors_df = pd.read_parquet(MODELS_DIR / "als_user_factors.parquet")
item_factors_df = pd.read_parquet(MODELS_DIR / "als_item_factors.parquet")

# Build lookup: idx -> factor vector
user_factors = {row["id"]: np.array(row["features"]) for _, row in user_factors_df.iterrows()}
item_factors = {row["id"]: np.array(row["features"]) for _, row in item_factors_df.iterrows()}

print(f"User factors loaded: {len(user_factors)}")
print(f"Item factors loaded: {len(item_factors)}")

def predict_als(user_idx, movie_idx):
    if user_idx not in user_factors or movie_idx not in item_factors:
        return global_mean
    return float(np.dot(user_factors[user_idx], item_factors[movie_idx]))

# Quick sanity check on a few test rows
sample = test_df.sample(5, random_state=1)
for row in sample.itertuples(index=False):
    pred = predict_als(row.user_idx, row.movie_idx)
    print(f"user={row.user_idx}, movie={row.movie_idx}, actual={row.rating}, predicted={pred:.2f}")

User factors loaded: 6040
Item factors loaded: 3667
user=5808, movie=1653, actual=1, predicted=1.65
user=1018, movie=2959, actual=4, predicted=2.74
user=5637, movie=1966, actual=3, predicted=2.40
user=3610, movie=3550, actual=5, predicted=4.03
user=1119, movie=1668, actual=3, predicted=3.57


In [3]:
def get_top_n_recommendations(user_idx, n=10, exclude_seen=None):
    """Score all movies for a user and return top-N movie_idx."""
    if user_idx not in user_factors:
        return []
    u_vec = user_factors[user_idx]
    scores = {}
    for m_idx, i_vec in item_factors.items():
        if exclude_seen and m_idx in exclude_seen:
            continue
        scores[m_idx] = np.dot(u_vec, i_vec)
    top_n = sorted(scores.items(), key=lambda x: -x[1])[:n]
    return [m for m, s in top_n]


def dcg_at_k(relevances, k):
    relevances = np.array(relevances)[:k]
    if len(relevances) == 0:
        return 0.0
    return np.sum(relevances / np.log2(np.arange(2, len(relevances) + 2)))


def ndcg_at_k(recommended, relevant_set, k):
    relevances = [1 if item in relevant_set else 0 for item in recommended[:k]]
    dcg = dcg_at_k(relevances, k)
    ideal_relevances = sorted(relevances, reverse=True)
    idcg = dcg_at_k(ideal_relevances, k)
    return dcg / idcg if idcg > 0 else 0.0


# Build per-user seen (train) and relevant (test, rating>=4) sets
train_seen = train_df.groupby("user_idx")["movie_idx"].apply(set)
test_relevant = (
    test_df[test_df["rating"] >= 4]
    .groupby("user_idx")["movie_idx"]
    .apply(set)
)

# Evaluate on a sample of users (full 6040 users x 3667 movies scoring is slow in pure Python)
np.random.seed(42)
eval_users = np.random.choice(test_relevant.index, size=300, replace=False)

precisions, recalls, ndcgs, hits = [], [], [], []
K = 10

for u in eval_users:
    relevant = test_relevant.get(u, set())
    if len(relevant) == 0:
        continue
    seen = train_seen.get(u, set())
    recs = get_top_n_recommendations(u, n=K, exclude_seen=seen)
    if not recs:
        continue

    hit_count = len(set(recs) & relevant)
    precisions.append(hit_count / K)
    recalls.append(hit_count / len(relevant))
    ndcgs.append(ndcg_at_k(recs, relevant, K))
    hits.append(1 if hit_count > 0 else 0)

print(f"Evaluated on {len(precisions)} users")
print(f"Precision@{K}: {np.mean(precisions):.4f}")
print(f"Recall@{K}: {np.mean(recalls):.4f}")
print(f"NDCG@{K}: {np.mean(ndcgs):.4f}")
print(f"Hit Rate@{K}: {np.mean(hits):.4f}")

Evaluated on 300 users
Precision@10: 0.0207
Recall@10: 0.0100
NDCG@10: 0.0619
Hit Rate@10: 0.1533
